# 03 - Data Cleaning

* Resolve the data issues identified in Notebook 02 and save the cleaned datasets as CSV files.

In [1]:
import pandas as pd

customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
order_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
products = pd.read_csv('../data/raw/olist_products_dataset.csv')
sellers = pd.read_csv('../data/raw/olist_sellers_dataset.csv')
category_translation = pd.read_csv('../data/raw/product_category_name_translation.csv')


## 1. Fix date columns
* Convert date fields from text format into proper date/time format.

In [2]:
date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'])
reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'])
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'])

orders.dtypes[date_cols]


order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

## 2. Remove duplicates
* Remove duplicate records to ensure the data contains unique entries.

In [3]:
customers = customers.drop_duplicates()
orders = orders.drop_duplicates()
order_items = order_items.drop_duplicates()
payments = payments.drop_duplicates()
products = products.drop_duplicates(subset=['product_id'])
sellers = sellers.drop_duplicates(subset=['seller_id'])

# keep only the latest review per order, then remove any leftover duplicate review_id
reviews = reviews.sort_values('review_answer_timestamp')
reviews = reviews.drop_duplicates(subset=['order_id'], keep='last')
reviews = reviews.drop_duplicates(subset=['review_id'], keep='last')

print('orders left:', len(orders))
print('reviews left:', len(reviews))


orders left: 99441
reviews left: 98095


## 3. Fill missing values
* Replace missing values appropriately based on the column and its meaning.

In [4]:
reviews['review_comment_message'] = reviews['review_comment_message'].fillna('')
reviews['review_comment_title'] = reviews['review_comment_title'].fillna('')

products['product_category_name'] = products['product_category_name'].fillna('unknown')

# fill missing weight/size with the average for that category
for col in ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']:
    products[col] = products[col].fillna(products[col].mean())


## 4. Standardize text and translate category names
* Clean text values and convert product category names into English.

In [5]:
customers['customer_city'] = customers['customer_city'].str.strip().str.lower()
sellers['seller_city'] = sellers['seller_city'].str.strip().str.lower()

# zip codes should keep leading zeros
customers['customer_zip_code_prefix'] = customers['customer_zip_code_prefix'].astype(str).str.zfill(5)
sellers['seller_zip_code_prefix'] = sellers['seller_zip_code_prefix'].astype(str).str.zfill(5)

# add English category names
products = products.merge(category_translation, on='product_category_name', how='left')
products['product_category_name_english'] = products['product_category_name_english'].fillna('unknown')
products = products.drop(columns=['product_category_name'])
products = products.rename(columns={'product_category_name_english': 'product_category_name'})

# keep only the columns needed for the database
products = products[['product_id', 'product_category_name', 'product_weight_g',
                      'product_length_cm', 'product_height_cm', 'product_width_cm']]

products.head()


,product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumery,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,art,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,sports_leisure,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,baby,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,housewares,625.0,20.0,17.0,13.0


## 5. Save cleaned data

In [6]:
customers.to_csv('../data/cleaned/customers_cleaned.csv', index=False)
orders.to_csv('../data/cleaned/orders_cleaned.csv', index=False)
order_items.to_csv('../data/cleaned/order_items_cleaned.csv', index=False)
payments.to_csv('../data/cleaned/payments_cleaned.csv', index=False)
reviews.to_csv('../data/cleaned/reviews_cleaned.csv', index=False)
products.to_csv('../data/cleaned/products_cleaned.csv', index=False)
sellers.to_csv('../data/cleaned/sellers_cleaned.csv', index=False)

print('All cleaned files saved to data/cleaned/')


All cleaned files saved to data/cleaned/


# Summary

* Date fields were corrected, duplicates removed, and missing values handled.
* Product category names were translated into English.
* 7 cleaned CSV files were saved for MySQL loading in Notebook 04.

